## 02. Preparing Count Data for Modelling


In this notebook, we prepare the AWV bicycle count data for the first stage of the project: modelling normal cycling traffic in Flanders.

The raw data are provided as monthly CSV files with 15-minute traffic counts. We focus only on cyclist observations (`FIETSERS`) and aggregate the data to 2-hour intervals. This reduces short-term noise while still preserving daily traffic patterns such as morning and evening peaks.

We also keep track of missing count values during aggregation. Missing counts are not treated as zero, because a missing value means that the count is unknown, while zero means that the sensor recorded no cyclists. For each 2-hour interval, we record how many 15-minute observations were available.

After cleaning and aggregation, we add site information and calendar variables such as public holidays and school holidays, as well as fuel prices data.

Important: We included fuel prices data into the normal counts modelling because  they are unlikely to explain sudden short-term deviations in one specific 2-hour interval. But they may influence the general expected level of cycling traffic over weeks or months, which should be accounted for when normal cycling counts are modelled.

After collecting data for potential factors for explaining deviations, it can also be included in this pre-processing step.

### Importing packages and defining path


In [1]:
import pandas as pd
from pathlib import Path

project_folder = Path("..")

raw_folder = project_folder / "data" / "raw"
processed_folder = project_folder / "data" / "processed"
external_folder = project_folder / "data" / "external"
diagnostics_folder = processed_folder / "diagnostics"


raw_folder.mkdir(parents=True, exist_ok=True)
processed_folder.mkdir(parents=True, exist_ok=True)
external_folder.mkdir(parents=True, exist_ok=True)
diagnostics_folder.mkdir(parents=True, exist_ok=True)

### File names and colum names

In [2]:
metadata_files = [
    "sites.csv",
    "richtingen.csv",
]

count_files = [
    "data-2022-04.csv",
    "data-2022-05.csv",
    "data-2022-06.csv",
    "data-2022-07.csv",
    "data-2022-08.csv",
    "data-2022-09.csv",
    "data-2022-10.csv",
    "data-2022-11.csv",
    "data-2022-12.csv",

    "data-2023-01.csv",
    "data-2023-02.csv",
    "data-2023-03.csv",
    "data-2023-04.csv",
    "data-2023-05.csv",
    "data-2023-06.csv",
    "data-2023-07.csv",
    "data-2023-08.csv",
    "data-2023-09.csv",
    "data-2023-10.csv",
    "data-2023-11.csv",
    "data-2023-12.csv",

    "data-2024-01.csv",
    "data-2024-02.csv",
    "data-2024-03.csv",
    "data-2024-04.csv",
    "data-2024-05.csv",
    "data-2024-06.csv",
    "data-2024-07.csv",
    "data-2024-08.csv",
    "data-2024-09.csv",
    "data-2024-10.csv",
    "data-2024-11.csv",
    "data-2024-12.csv",

    "data-2025-01.csv",
    "data-2025-02.csv",
    "data-2025-03.csv",
    "data-2025-04.csv",
    "data-2025-05.csv",
    "data-2025-06.csv",
    "data-2025-07.csv",
    "data-2025-08.csv",
    "data-2025-09.csv",
    "data-2025-10.csv",
    "data-2025-11.csv",
    "data-2025-12.csv",

    "data-2026-01.csv",
    "data-2026-02.csv",
    "data-2026-03.csv",
    "data-2026-04.csv",
]



In [3]:
site_columns = [
    "site_id",
    "site_number",
    "longitude",
    "latitude",
    "site_name",
    "domain",
    "road_number",
    "district",
    "municipality",
    "counting_interval_minutes",
    "installation_date",
]

direction_columns = [
    "site_id",
    "direction",
    "direction_description",
]

count_columns = [
    "site_id",
    "direction",
    "traffic_type",
    "start_time",
    "end_time",
    "count",
]

### Reading metadata and external data for modelling normal counts

In [4]:
sites = pd.read_csv(
    raw_folder / "sites.csv",
    header=None,
    names=site_columns,
)

directions = pd.read_csv(
    raw_folder / "richtingen.csv",
    header=None,
    names=direction_columns,
)

public_holidays = pd.read_csv(
    external_folder / "public_holidays.csv"
)

school_holidays_flanders = pd.read_csv(
    external_folder / "school_holidays_flanders.csv"
)

fuel_prices = pd.read_csv(
    external_folder / "fuel_prices_belgium.csv"
)

Converting dates into date format

In [5]:
sites["installation_date"] = pd.to_datetime(
    sites["installation_date"],
    errors="coerce"
)

public_holidays["date"] = pd.to_datetime(
    public_holidays["date"],
    errors="coerce"
)

school_holidays_flanders["start_date"] = pd.to_datetime(
    school_holidays_flanders["start_date"],
    errors="coerce"
)

school_holidays_flanders["end_date"] = pd.to_datetime(
    school_holidays_flanders["end_date"],
    errors="coerce"
)

fuel_prices["date"] = pd.to_datetime(
    fuel_prices["date"],
    errors="coerce"
)

In [6]:
print("Sites:", sites.shape)
print("Directions:", directions.shape)
print("Public holidays:", public_holidays.shape)
print("School holidays:", school_holidays_flanders.shape)
print("Fuel prices:", fuel_prices.shape)

Sites: (151, 11)
Directions: (305, 3)
Public holidays: (59, 2)
School holidays: (21, 3)
Fuel prices: (212, 2)


### Missing values in the monthly count data among the observations recorded for cyclists

During the data loading procedures, missing values were detected for counts in the monthly counts dataset. We are trying to explore whether there are some particluar patterns by month and site associated with missing values.

In [7]:
missing_counts_by_month = []

for file_name in count_files:
    df = pd.read_csv(
        raw_folder / file_name,
        header=None,
        names=count_columns,
    )

    
    df = df[df["traffic_type"] == "FIETSERS"].copy()
    df["count"] = pd.to_numeric(df["count"], errors="coerce")

    missing_counts_by_month.append({
        "file_name": file_name,
        "rows": len(df),
        "missing_count": df["count"].isna().sum(),
        "missing_count_share": df["count"].isna().mean(),
        "zero_count": (df["count"] == 0).sum(),
        "zero_count_share": (df["count"] == 0).mean(),
    })

missing_counts_by_month = pd.DataFrame(missing_counts_by_month)

missing_counts_by_month

,file_name,rows,missing_count,missing_count_share,zero_count,zero_count_share
0,data-2022-04.csv,267250,63154,0.236311,111497,0.417201
1,data-2022-05.csv,368518,90142,0.244607,147501,0.400255
2,data-2022-06.csv,486830,124142,0.255001,189375,0.388996
3,data-2022-07.csv,632122,117048,0.185167,267164,0.422646
4,data-2022-08.csv,668966,34446,0.051491,324661,0.485318
5,data-2022-09.csv,714104,65592,0.091852,359786,0.503829
6,data-2022-10.csv,767808,29656,0.038624,403696,0.525777
7,data-2022-11.csv,740358,6,0.000008,436504,0.589585
8,data-2022-12.csv,763118,1494,0.001958,492127,0.644890
9,data-2023-01.csv,780396,17086,0.021894,484211,0.620468


In [8]:
missing_counts_by_month_by_site = []

for file_name in count_files:
    df = pd.read_csv(
        raw_folder / file_name,
        header=None,
        names=count_columns,
    )

    df = df[df["traffic_type"] == "FIETSERS"].copy()
    df["count"] = pd.to_numeric(df["count"], errors="coerce")

    grouped = df.groupby("site_id")["count"]
    group = grouped.agg(
        total_rows="size",
        missing_count=lambda x: x.isna().sum(),
        zero_count=lambda x: (x == 0).sum(),
    ).reset_index()

    group["missing_share"] = group["missing_count"] / group["total_rows"]
    group["zero_share"] = group["zero_count"] / group["total_rows"]
    group["file_name"] = file_name

    missing_counts_by_month_by_site.append(group)

missing_counts_by_month_by_site = pd.concat(missing_counts_by_month_by_site, ignore_index=True)
missing_counts_by_month_by_site

,site_id,total_rows,missing_count,zero_count,missing_share,zero_share,file_name
0,1,5760,0,2492,0.0,0.432639,data-2022-04.csv
1,2,5760,0,3138,0.0,0.544792,data-2022-04.csv
2,3,5760,0,3035,0.0,0.526910,data-2022-04.csv
3,4,5760,0,4286,0.0,0.744097,data-2022-04.csv
4,5,5760,0,4095,0.0,0.710938,data-2022-04.csv
...,...,...,...,...,...,...,...
6538,148,5760,0,2487,0.0,0.431771,data-2026-04.csv
6539,149,5760,0,2192,0.0,0.380556,data-2026-04.csv
6540,150,5760,0,3597,0.0,0.624479,data-2026-04.csv
6541,151,5760,0,3063,0.0,0.531771,data-2026-04.csv


In [9]:
missing_counts_by_month_by_site = missing_counts_by_month_by_site.sort_values(
    "missing_share",
    ascending=False
)

missing_counts_by_month_by_site.head(300)

,site_id,total_rows,missing_count,zero_count,missing_share,zero_share,file_name
4377,119,5952,5952,0,1.000000,0.000000,data-2025-01.csv
109,63,1328,1328,0,1.000000,0.000000,data-2022-05.csv
111,65,1326,1326,0,1.000000,0.000000,data-2022-05.csv
112,66,352,352,0,1.000000,0.000000,data-2022-05.csv
113,67,354,354,0,1.000000,0.000000,data-2022-05.csv
...,...,...,...,...,...,...,...
175,61,5760,384,3253,0.066667,0.564757,data-2022-06.csv
2098,52,5760,384,988,0.066667,0.171528,data-2023-09.csv
5568,41,5952,384,3503,0.064516,0.588542,data-2025-10.csv
429,106,5952,384,4187,0.064516,0.703461,data-2022-08.csv


In [10]:
missing_counts_by_month_by_site.to_csv(
    diagnostics_folder / "missing_counts_by_site.csv",
    index=False
)

***Possible way to deal with missing values***

Given the distribution of the missing values across months and sites, we try to handle missing values during aggregation from 15-minute intervals to 2-hour intervals. For each 2-hour interval, we store the number of observed and missing 15-minute observations. Intervals with at least 5 out of the expected 8 observations are retained for modelling. For these mostly complete intervals, an adjusted count is computed by scaling the observed sum to an 8-interval equivalent. Intervals with fewer than 5 observed observations are excluded from the modelling dataset.

This approach avoids imputing fully missing site-periods while also avoiding unnecessary loss of mostly complete 2-hour intervals.

We can also test sensitivity to a 5/8 threshold choice.


### Intervals check

In [11]:
actual_interval_values = []

for file_name in count_files:
    df = pd.read_csv(
        raw_folder / file_name,
        header=None,
        names=count_columns,
    )

    df = df[df["traffic_type"] == "FIETSERS"].copy()

    df["start_time"] = pd.to_datetime(df["start_time"], errors="coerce")
    df["end_time"] = pd.to_datetime(df["end_time"], errors="coerce")

    df["actual_interval_minutes"] = (
        (df["end_time"] - df["start_time"])
        .dt.total_seconds()
        / 60
    )

    actual_interval_values.append(df[["actual_interval_minutes"]])

actual_interval_values = pd.concat(actual_interval_values, ignore_index=True)

actual_interval_values["actual_interval_minutes"].value_counts(dropna=False).sort_index()

actual_interval_minutes
15.0    37909558
75.0         274
Name: count, dtype: int64

Not all observation intervals are of 15 minutes length.

In [12]:
incorrect_intervals = []

for file_name in count_files:
    df = pd.read_csv(
        raw_folder / file_name,
        header=None,
        names=count_columns,
    )

    df = df[df["traffic_type"] == "FIETSERS"].copy()
    df["start_time"] = pd.to_datetime(df["start_time"], errors="coerce")
    df["end_time"] = pd.to_datetime(df["end_time"], errors="coerce")
    df["count"] = pd.to_numeric(df["count"], errors="coerce")

    df["actual_interval_minutes"] = (
        (df["end_time"] - df["start_time"]).dt.total_seconds() / 60
    )

    incorrect = df[(df["actual_interval_minutes"] - 15).abs() > 0.1].copy()
    incorrect["file_name"] = file_name

    incorrect_intervals.append(
        incorrect[
            [
                "file_name",
                "site_id",
                "direction",
                "start_time",
                "end_time",
                "actual_interval_minutes",
                "count",
            ]
        ]
    )

incorrect_intervals = pd.concat(incorrect_intervals, ignore_index=True)
incorrect_intervals.head(50)

,file_name,site_id,direction,start_time,end_time,actual_interval_minutes,count
0,data-2023-03.csv,1,IN,2023-03-26 01:45:00,2023-03-26 03:00:00,75.0,0.0
1,data-2023-03.csv,1,OUT,2023-03-26 01:45:00,2023-03-26 03:00:00,75.0,0.0
2,data-2023-03.csv,2,IN,2023-03-26 01:45:00,2023-03-26 03:00:00,75.0,1.0
3,data-2023-03.csv,2,OUT,2023-03-26 01:45:00,2023-03-26 03:00:00,75.0,0.0
4,data-2023-03.csv,3,IN,2023-03-26 01:45:00,2023-03-26 03:00:00,75.0,0.0
5,data-2023-03.csv,3,OUT,2023-03-26 01:45:00,2023-03-26 03:00:00,75.0,1.0
6,data-2023-03.csv,4,IN,2023-03-26 01:45:00,2023-03-26 03:00:00,75.0,0.0
7,data-2023-03.csv,4,OUT,2023-03-26 01:45:00,2023-03-26 03:00:00,75.0,0.0
8,data-2023-03.csv,5,IN,2023-03-26 01:45:00,2023-03-26 03:00:00,75.0,0.0
9,data-2023-03.csv,5,OUT,2023-03-26 01:45:00,2023-03-26 03:00:00,75.0,0.0


In [13]:
incorrect_intervals.to_csv(
    diagnostics_folder / "incorrect_intervals.csv",
    index=False
)

**Important**: There was one recording disruption that happened on 26.03.2023 when all the data between 01:45:00 and 03:00:00 was recorded as a single interval due to daylight saving time changes.

### Checking all spring and autumn DST days

In [14]:
spring_dst_dates = [
    "2023-03-26",
    "2024-03-31",
    "2025-03-30",
    "2026-03-29",
]

spring_dst_check = []

for file_name in count_files:
    df = pd.read_csv(
        raw_folder / file_name,
        header=None,
        names=count_columns,
    )

    df = df[df["traffic_type"] == "FIETSERS"].copy()

    df["start_time"] = pd.to_datetime(df["start_time"], errors="coerce")
    df["end_time"] = pd.to_datetime(df["end_time"], errors="coerce")
    df["count"] = pd.to_numeric(df["count"], errors="coerce")

    df["date"] = df["start_time"].dt.floor("D")
    df["hour"] = df["start_time"].dt.hour
    df["minute"] = df["start_time"].dt.minute

    df["actual_interval_minutes"] = (
        (df["end_time"] - df["start_time"])
        .dt.total_seconds()
        / 60
    )

    temp = df[df["date"].isin(pd.to_datetime(spring_dst_dates))].copy()
    temp["file_name"] = file_name

    spring_dst_check.append(temp)

spring_dst_check = pd.concat(spring_dst_check, ignore_index=True)

In [15]:
spring_dst_hour_summary = (
    spring_dst_check
    .groupby(["date", "hour"])
    .agg(
        rows=("count", "size"),
        unique_sites=("site_id", "nunique"),
        unique_intervals=("actual_interval_minutes", lambda x: sorted(x.dropna().unique())),
        missing_counts=("count", lambda x: x.isna().sum()),
    )
    .reset_index()
)

spring_dst_hour_summary

,date,hour,rows,unique_sites,unique_intervals,missing_counts
0,2023-03-26,0,1096,137,[15.0],24
1,2023-03-26,1,1096,137,"[15.0, 75.0]",24
2,2023-03-26,3,1096,137,[15.0],24
3,2023-03-26,4,1096,137,[15.0],24
4,2023-03-26,5,1096,137,[15.0],24
...,...,...,...,...,...,...
90,2026-03-29,19,1160,145,[15.0],6
91,2026-03-29,20,1160,145,[15.0],6
92,2026-03-29,21,1160,145,[15.0],6
93,2026-03-29,22,1160,145,[15.0],6


In [16]:
spring_dst_hour_summary.to_csv(
    diagnostics_folder / "spring_dst_hour_summary.csv",
    index=False
)

The spring daylight-saving transition is represented inconsistently across years. In 2023, the transition appears as a 75-minute interval from 01:45 to 03:00, while in later years the skipped 02:00 hour is present but contains missing counts. 

In [17]:
autumn_dst_dates = [
    "2022-10-30",
    "2023-10-29",
    "2024-10-27",
    "2025-10-26",
]

autumn_dst_check = []

for file_name in count_files:
    df = pd.read_csv(
        raw_folder / file_name,
        header=None,
        names=count_columns,
    )

    df = df[df["traffic_type"] == "FIETSERS"].copy()

    df["start_time"] = pd.to_datetime(df["start_time"], errors="coerce")
    df["end_time"] = pd.to_datetime(df["end_time"], errors="coerce")
    df["count"] = pd.to_numeric(df["count"], errors="coerce")

    df["date"] = df["start_time"].dt.floor("D")

    df["actual_interval_minutes"] = (
        (df["end_time"] - df["start_time"])
        .dt.total_seconds()
        / 60
    )

    temp = df[df["date"].isin(pd.to_datetime(autumn_dst_dates))].copy()
    temp["file_name"] = file_name

    autumn_dst_check.append(temp)

autumn_dst_check = pd.concat(autumn_dst_check, ignore_index=True)

In [18]:
autumn_dst_check["is_duplicate_start_time"] = autumn_dst_check.duplicated(
    subset=["date", "site_id", "direction", "start_time"],
    keep=False
)

In [19]:
autumn_dst_duplicates = autumn_dst_check[
    autumn_dst_check["is_duplicate_start_time"]
].copy()

autumn_dst_duplicates = autumn_dst_duplicates.sort_values(
    ["date", "site_id", "direction", "start_time", "end_time"]
)

autumn_dst_duplicates

,site_id,direction,traffic_type,start_time,end_time,count,date,actual_interval_minutes,file_name,is_duplicate_start_time


In [20]:
autumn_dst_check["hour"] = autumn_dst_check["start_time"].dt.hour

autumn_dst_hour_summary = (
    autumn_dst_check
    .groupby(["date", "hour"])
    .agg(
        rows=("count", "size"),
        unique_sites=("site_id", "nunique"),
        unique_intervals=("actual_interval_minutes", lambda x: sorted(x.dropna().unique())),
        missing_counts=("count", lambda x: x.isna().sum()),
        duplicate_start_time_rows=("is_duplicate_start_time", "sum"),
    )
    .reset_index()
)

autumn_dst_hour_summary.to_csv(
    diagnostics_folder / "autumn_dst_hour_summary.csv",
    index=False
)

autumn_dst_hour_summary

,date,hour,rows,unique_sites,unique_intervals,missing_counts,duplicate_start_time_rows
0,2022-10-30,0,1032,129,[15.0],0,0
1,2022-10-30,1,1032,129,[15.0],0,0
2,2022-10-30,2,1032,129,[15.0],0,0
3,2022-10-30,3,1032,129,[15.0],0,0
4,2022-10-30,4,1032,129,[15.0],0,0
...,...,...,...,...,...,...,...
91,2025-10-26,19,1160,145,[15.0],22,0
92,2025-10-26,20,1160,145,[15.0],22,0
93,2025-10-26,21,1160,145,[15.0],22,0
94,2025-10-26,22,1160,145,[15.0],22,0


The autumn daylight saving time transition does not produce duplicated local timestamps in the AWV cyclist count data. All intervals on autumn DST dates remain 15 minutes long, and no duplicated combinations of date, site, direction and start time were found.

## Cleaning and agregating the count data


Only standard 15-minute count observations are used for aggregation. A very small number of non-standard intervals occurs around the spring daylight saving time transition, including one 75-minute interval in 2023. 
These rows are excluded before aggregation because they are not comparable to regular 15-minute observations.


In [21]:
def aggregate_monthly_counts(file_name):

    df = pd.read_csv(
        raw_folder / file_name,
        header=None,
        names=count_columns,
    )

    df = df[df["traffic_type"] == "FIETSERS"].copy()

    df["start_time"] = pd.to_datetime(df["start_time"], errors="coerce")
    df["end_time"] = pd.to_datetime(df["end_time"], errors="coerce")
    df["count"] = pd.to_numeric(df["count"], errors="coerce")

    df["actual_interval_minutes"] = (
        (df["end_time"] - df["start_time"])
        .dt.total_seconds()
        / 60
    )
    
    df = df[df["actual_interval_minutes"] == 15].copy()

    df["year"] = df["start_time"].dt.year
    df["date"] = df["start_time"].dt.floor("D")
    df["month"] = df["start_time"].dt.month
    df["hour"] = df["start_time"].dt.hour
    df["hour_bin"] = (df["hour"] // 2) * 2
    df["weekday"] = df["start_time"].dt.day_name()

    df_aggregated = (
        df
        .groupby(
            [
                "site_id",
                "direction",
                "year",
                "date",
                "month",
                "weekday",
                "hour_bin",
            ],
            as_index=False
        )
        .agg(
            count=("count", lambda x: x.sum(min_count=1)),
            observed_intervals=("count", "count"),
            total_intervals=("count", "size"),
            missing_intervals=("count", lambda x: x.isna().sum()),
        )
    )

    df_aggregated["missing_share"] = (
        df_aggregated["missing_intervals"] / df_aggregated["total_intervals"]
    )

    return df_aggregated

In [22]:
aggregated_monthly_data = []

for file_name in count_files:
    monthly_aggregated = aggregate_monthly_counts(file_name)
    aggregated_monthly_data.append(monthly_aggregated)

counts_aggregated = pd.concat(aggregated_monthly_data, ignore_index=True)

### Removing 2-hour intervals with less than 5 component 15-min intervals

In [23]:
minimum_observed_intervals = 5

counts_model = counts_aggregated.copy()

counts_model = counts_model[counts_model["count"].notna()].copy()

counts_model = counts_model[
    counts_model["observed_intervals"] >= minimum_observed_intervals
].copy()


### Rescalling incomplete intervals

In [24]:
counts_model["expected_intervals_for_row"] = 8

counts_model["count_rescaled"] = (
    counts_model["count"]
    * counts_model["expected_intervals_for_row"]
    / counts_model["observed_intervals"]
)

counts_model["rescaled"] = (
    counts_model["observed_intervals"] < counts_model["expected_intervals_for_row"]
).astype(int)


### Checks

How many rows were kept?

In [25]:
print("Rows before filtering:", counts_aggregated.shape[0])
print("Rows after filtering:", counts_model.shape[0])
print("Rows removed:", counts_aggregated.shape[0] - counts_model.shape[0])

print(
    "Share removed:",
    round(
        (counts_aggregated.shape[0] - counts_model.shape[0])
        / counts_aggregated.shape[0],
        4
    )
)

Rows before filtering: 4738968
Rows after filtering: 4613248
Rows removed: 125720
Share removed: 0.0265


Distribution of observed intervals after filtering

In [26]:
counts_model["observed_intervals"].value_counts().sort_index()

observed_intervals
5         65
6         57
7        434
8    4612692
Name: count, dtype: int64

How many rows were rescaled?

In [27]:
counts_model["rescaled"].value_counts()

rescaled
0    4612692
1        556
Name: count, dtype: int64

Rescalling should not change the complete rows

In [28]:
complete_rows_changed = counts_model[
    (counts_model["observed_intervals"] == 8)
    & (counts_model["count"] != counts_model["count_rescaled"])
]

complete_rows_changed.shape[0]

0

### Adding matadata and external source information

In [29]:
def add_metadata_and_external_data(
    counts_model,
    sites,
    directions,
    public_holidays,
    school_holidays_flanders,
    fuel_prices,
):
    counts_model_final = counts_model.copy()

    #Site metadata
    site_variables = [
        "site_id",
        "longitude",
        "latitude",
        "site_name",
        "municipality",
        "district",
        "installation_date",
    ]

    counts_model_final = counts_model_final.merge(
        sites[site_variables],
        on="site_id",
        how="left",
    )

    #Direction metadata
    direction_variables = [
        "site_id",
        "direction",
        "direction_description",
    ]

    counts_model_final = counts_model_final.merge(
        directions[direction_variables],
        on=["site_id", "direction"],
        how="left",
    )

    #Public holidays
    public_holidays_for_merge = public_holidays.copy()

    public_holidays_for_merge["is_public_holiday"] = 1

    public_holidays_for_merge = public_holidays_for_merge[
        ["date", "is_public_holiday", "holiday_name"]
    ].copy()

    counts_model_final = counts_model_final.merge(
        public_holidays_for_merge,
        on="date",
        how="left",
    )

    counts_model_final["is_public_holiday"] = (
        counts_model_final["is_public_holiday"]
        .fillna(0)
        .astype(int)
    )

    counts_model_final["holiday_name"] = (
        counts_model_final["holiday_name"]
        .fillna("No public holiday")
    )

    #School holidays
    counts_model_final["is_school_holiday"] = 0
    counts_model_final["school_holiday_name"] = "No school holiday"

    for _, row in school_holidays_flanders.iterrows():
        in_school_holiday = (
            (counts_model_final["date"] >= row["start_date"])
            & (counts_model_final["date"] <= row["end_date"])
        )

        counts_model_final.loc[
            in_school_holiday,
            "is_school_holiday"
        ] = 1

        counts_model_final.loc[
            in_school_holiday,
            "school_holiday_name"
        ] = row["school_holiday_name"]

    #Fuel prices
    fuel_prices_for_merge = fuel_prices.copy()

    fuel_prices_for_merge = fuel_prices_for_merge.rename(
        columns={
            "petrol_95_eur_per_1000_l": "fuel_price_petrol_95"
        }
    )

    fuel_prices_for_merge = fuel_prices_for_merge[
        ["date", "fuel_price_petrol_95"]
    ].copy()

    fuel_prices_for_merge = fuel_prices_for_merge.sort_values("date")

    #Create one row per calendar day in the modelling period
    daily_dates = pd.DataFrame({
        "date": pd.date_range(
            counts_model_final["date"].min(),
            counts_model_final["date"].max(),
            freq="D",
        )
    })

    #Assign each day the most recent available weekly fuel price
    daily_fuel_prices = pd.merge_asof(
        daily_dates.sort_values("date"),
        fuel_prices_for_merge.sort_values("date"),
        on="date",
        direction="backward",
    )

    #Dates are before the first fuel-price date filled  using the first available fuel price.
    daily_fuel_prices["fuel_price_petrol_95"] = (
        daily_fuel_prices["fuel_price_petrol_95"]
        .bfill()
    )

    counts_model_final = counts_model_final.merge(
        daily_fuel_prices,
        on="date",
        how="left",
    )

    counts_model_final = counts_model_final.sort_values(
        ["site_id", "direction", "date", "hour_bin"]
    ).reset_index(drop=True)

    return counts_model_final

In [30]:
counts_model_final = add_metadata_and_external_data(
    counts_model=counts_model,
    sites=sites,
    directions=directions,
    public_holidays=public_holidays,
    school_holidays_flanders=school_holidays_flanders,
    fuel_prices=fuel_prices,
)

In [31]:
counts_model_final.head(50)

,site_id,direction,year,date,month,weekday,hour_bin,count,observed_intervals,total_intervals,...,site_name,municipality,district,installation_date,direction_description,is_public_holiday,holiday_name,is_school_holiday,school_holiday_name,fuel_price_petrol_95
0,1,IN,2022,2022-04-01,4,Friday,0,0.0,8,8,...,Machelen,Machelen,AWV212,2019-08-22,Machelen Cyclists rich. Brucargo,0,No public holiday,0,No school holiday,1807.61
1,1,IN,2022,2022-04-01,4,Friday,2,0.0,8,8,...,Machelen,Machelen,AWV212,2019-08-22,Machelen Cyclists rich. Brucargo,0,No public holiday,0,No school holiday,1807.61
2,1,IN,2022,2022-04-01,4,Friday,4,6.0,8,8,...,Machelen,Machelen,AWV212,2019-08-22,Machelen Cyclists rich. Brucargo,0,No public holiday,0,No school holiday,1807.61
3,1,IN,2022,2022-04-01,4,Friday,6,9.0,8,8,...,Machelen,Machelen,AWV212,2019-08-22,Machelen Cyclists rich. Brucargo,0,No public holiday,0,No school holiday,1807.61
4,1,IN,2022,2022-04-01,4,Friday,8,9.0,8,8,...,Machelen,Machelen,AWV212,2019-08-22,Machelen Cyclists rich. Brucargo,0,No public holiday,0,No school holiday,1807.61
5,1,IN,2022,2022-04-01,4,Friday,10,4.0,8,8,...,Machelen,Machelen,AWV212,2019-08-22,Machelen Cyclists rich. Brucargo,0,No public holiday,0,No school holiday,1807.61
6,1,IN,2022,2022-04-01,4,Friday,12,5.0,8,8,...,Machelen,Machelen,AWV212,2019-08-22,Machelen Cyclists rich. Brucargo,0,No public holiday,0,No school holiday,1807.61
7,1,IN,2022,2022-04-01,4,Friday,14,8.0,8,8,...,Machelen,Machelen,AWV212,2019-08-22,Machelen Cyclists rich. Brucargo,0,No public holiday,0,No school holiday,1807.61
8,1,IN,2022,2022-04-01,4,Friday,16,9.0,8,8,...,Machelen,Machelen,AWV212,2019-08-22,Machelen Cyclists rich. Brucargo,0,No public holiday,0,No school holiday,1807.61
9,1,IN,2022,2022-04-01,4,Friday,18,5.0,8,8,...,Machelen,Machelen,AWV212,2019-08-22,Machelen Cyclists rich. Brucargo,0,No public holiday,0,No school holiday,1807.61


In [32]:
counts_model_final.to_csv(
    processed_folder / "counts_model_final.csv",
    index=False
)


### Checks

Rows and columns

In [33]:
print("counts_model rows:", counts_model.shape[0])
print("counts_model_final rows:", counts_model_final.shape[0])
print("columns:", counts_model_final.shape[1])

counts_model rows: 4613248
counts_model_final rows: 4613248
columns: 27


Duplicates

In [34]:
counts_model_final.duplicated(
    subset=["site_id", "direction", "date", "hour_bin"]
).sum()

np.int64(0)

Missing values

In [35]:
counts_model_final.isna().sum()

site_id                           0
direction                         0
year                              0
date                              0
month                             0
weekday                           0
hour_bin                          0
count                             0
observed_intervals                0
total_intervals                   0
missing_intervals                 0
missing_share                     0
expected_intervals_for_row        0
count_rescaled                    0
rescaled                          0
longitude                         0
latitude                          0
site_name                         0
municipality                   5914
district                      27064
installation_date                 0
direction_description             0
is_public_holiday                 0
holiday_name                      0
is_school_holiday                 0
school_holiday_name               0
fuel_price_petrol_95              0
dtype: int64

In [44]:
sites.isna().sum()

site_id                      0
site_number                  0
longitude                    0
latitude                     0
site_name                    0
domain                       0
road_number                  3
district                     3
municipality                 1
counting_interval_minutes    0
installation_date            0
dtype: int64

Public holidays

In [36]:
counts_model_final[
    counts_model_final["is_public_holiday"] == 1
][
    ["date", "holiday_name"]
].drop_duplicates().sort_values("date").head(20)

,date,holiday_name
168,2022-04-15,Good Friday
192,2022-04-17,Easter Sunday
204,2022-04-18,Easter Monday
360,2022-05-01,Labour Day
660,2022-05-26,Ascension Day
672,2022-05-27,Day after Ascension Day
792,2022-06-06,Whit Monday
1332,2022-07-21,Belgian National Day
1632,2022-08-15,Assumption Day
2561,2022-11-01,All Saints' Day


In [37]:
counts_model_final[
    counts_model_final["is_public_holiday"] == 0
]["holiday_name"].value_counts()

holiday_name
No public holiday    4437724
Name: count, dtype: int64

School holidays

In [38]:
counts_model_final[
    counts_model_final["is_school_holiday"] == 1
][
    ["date", "school_holiday_name"]
].drop_duplicates().sort_values("date").head(30)

,date,school_holiday_name
36,2022-04-04,Spring Holidays
48,2022-04-05,Spring Holidays
60,2022-04-06,Spring Holidays
72,2022-04-07,Spring Holidays
84,2022-04-08,Spring Holidays
96,2022-04-09,Spring Holidays
108,2022-04-10,Spring Holidays
120,2022-04-11,Spring Holidays
132,2022-04-12,Spring Holidays
144,2022-04-13,Spring Holidays


In [39]:
counts_model_final[
    counts_model_final["is_school_holiday"] == 0
]["school_holiday_name"].value_counts()

school_holiday_name
No school holiday    3295666
Name: count, dtype: int64

Fuel prices

In [41]:
counts_model_final[
    ["date", "fuel_price_petrol_95"]
].drop_duplicates().sort_values("date").head(50)

,date,fuel_price_petrol_95
0,2022-04-01,1807.61
12,2022-04-02,1807.61
24,2022-04-03,1807.61
36,2022-04-04,1807.61
48,2022-04-05,1807.61
60,2022-04-06,1807.61
72,2022-04-07,1807.61
84,2022-04-08,1807.61
96,2022-04-09,1807.61
108,2022-04-10,1807.61
